In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from autogluon.tabular import TabularPredictor

In [2]:
# PARAMETERS

# We are doing >= 4 opps as the positive class in the binary classification example
NUM_OPPS_FOR_TRAINING = 4

# Training filters (less than this and we don't trust the orbit)
MIN_ARC_LENGTH_FOR_TRAINING = 20
MIN_NUM_OBS_FOR_TRAINING = 16

# Total training time limit (seconds), don't set below 500, but potentially go up to two hours for maximum performance
TRAINING_TIME_LIMIT = 1000

# Bringing in Orbit Database and Specific Fixes/Filters on it

In [3]:
orb = pd.read_json("https://minorplanetcenter.net/Extended_Files/mpcorb_extended.json.gz",compression='gzip')
# MPC leaves the U parameter blank for some older objects.
# Any recent object should have a U, but some older ones without a well definied orbit are deemed lost and have a blank H.
# For our purposes we fill it with 10 here, even though some object with missing U actually have a much better defined orbit than U=9.
orb['U'] = pd.to_numeric(orb['U'], errors='coerce').fillna(10)
orb['H_MPC'] = orb['H']
orb = orb.convert_dtypes()
orb.drop(["Other_desigs"],axis=1,inplace=True)

In [4]:
# Corrected H for known photometry issues
corrections = pd.read_csv("absolute magnitude fixes.csv")
# update H where orb["Principal_desig"] = corrections["Object"] by setting "H" = "Corrected H"
for _, row in corrections.iterrows():
    orb.loc[orb["Principal_desig"] == row["Object"], "H"] = row["Corrected H"]

# Filter based on filter lists
filter_until_further_notice = pd.read_csv("filter until further notice.csv")
filter_out_unless_updated = pd.read_csv("filter out unless updated.csv")

# Apply filters
orb = orb[~orb["Principal_desig"].isin(filter_until_further_notice["Object"])]
# for the filter unless updated we filter only if Object = Principal_desig and arc_length = Arc_length
for _, row in filter_out_unless_updated.iterrows():
    orb = orb[~((orb["Principal_desig"] == row["Object"]) & (orb["Arc_length"] == row["Arc_length"]))]

# Feature Engineering

In [5]:
def feature_engineering(orb):
    r = orb['a']*(1+orb['e']/2)
    def add_vis(orb,param):
        orb['vis_'+str(param)] = (5*np.log10(np.maximum(r,1)*(np.maximum(r,1)-param))+orb['H']).astype(float)
        orb['visq_'+str(param)] = (5*np.log10(np.maximum(orb["Perihelion_dist"],1)*(np.maximum(orb["Perihelion_dist"],1)-param))+orb['H']).astype(float)

    # These just happen to be ones that seem to work well, but it is open to tuning.
    add_vis(orb,0)

    # Feature engineer orbital period syncing up with earth.
    # Get the absolute value of the difference between Orbital_period and it's nearest integer multiple of Earth's orbital period (1 year)
    orb['orbital_period_sync'] = np.abs(orb['Orbital_period'] - np.round(orb['Orbital_period']))

    # Unit vector (direction) of the point of perihelion in the Heliocentric Ecliptic Coordinate System
    perihelion_directions = np.array([
        np.cos(np.radians(orb["Node"])) * np.cos(np.radians(orb["Peri"])) - np.sin(np.radians(orb["Node"])) * np.sin(np.radians(orb["Peri"])) * np.cos(np.radians(orb["i"])),
        np.sin(np.radians(orb["Node"])) * np.cos(np.radians(orb["Peri"])) + np.cos(np.radians(orb["Node"])) * np.sin(np.radians(orb["Peri"])) * np.cos(np.radians(orb["i"])),
        np.sin(np.radians(orb["Peri"])) * np.sin(np.radians(orb["i"]))
    ])

    orb["Perihelion_direction_x"] = perihelion_directions[0]
    orb["Perihelion_direction_y"] = perihelion_directions[1]
    orb["Perihelion_direction_z"] = perihelion_directions[2]

    # It turns out that the perihelion direction matters not at all for e=0, so we make the length of these vectors proportional to the eccentricity
    # The model prefers these over the prior three
    orb["Perihelion_direction_x_e"] = orb["Perihelion_direction_x"]*orb["e"]
    orb["Perihelion_direction_y_e"] = orb["Perihelion_direction_y"]*orb["e"]
    orb["Perihelion_direction_z_e"] = orb["Perihelion_direction_z"]*orb["e"]

    orb["TJ"] = 5.203/orb["a"] + 2*np.cos(np.radians(orb["i"])) * np.sqrt(orb["a"]/5.203*(1-orb["e"]**2))

    # Calculate the angle between the galactic plane and an orbital plane using the dot product
    n_gal = np.array([-0.8676, 0.0104, 0.4971])
    n_ast = [np.sin(np.radians(orb["i"])) * np.sin(np.radians(orb["Node"])), -np.sin(np.radians(orb["i"])) * np.cos(np.radians(orb["Node"])), np.cos(np.radians(orb["i"]))]
    orb["gal_dist"] = np.degrees(np.arccos(n_gal[0]*n_ast[0] + n_gal[1]*n_ast[1] + n_gal[2]*n_ast[2]))

    def add_advanced_vis_features(orb, param=0.0):
        """
        Adds variations of visibility metrics based on different orbital geometries.
        param: A float offset subtracted from the geocentric distance term (r - param).
            Positive params bring the 'observer' closer; negative pushes them away.
        """
        
        # Pre-calculate common terms to avoid redundancy
        # Clip e to avoid division by zero or invalid math for hyperbolic orbits (just in case)
        e = np.clip(orb['e'], 0, 0.999) 
        a = orb['a']
        H = orb['H']
        # Convert angles to radians for numpy
        peri_rad = np.radians(orb['Peri'])
        
        # ---------------------------------------------------------
        # VARIATION 1: Aphelion Visibility (The "Worst Case")
        # r = Q = a * (1 + e)
        # ---------------------------------------------------------
        r_Q = a * (1 + e)
        # Geocentric proxy: We assume object is at opposition near aphelion
        delta_Q = np.maximum(r_Q - 1.0 - param, 0.001) # Approx distance from Earth
        
        col_name_Q = f'vis_Q_{param}'
        orb[col_name_Q] = (5 * np.log10(np.maximum(r_Q, 0.001) * delta_Q) + H).astype(float)

        # ---------------------------------------------------------
        # VARIATION 2: Semi-Latus Rectum (The "Geometric Shoulder")
        # r = p = a * (1 - e^2)
        # This is the distance at True Anomaly = 90 degrees
        # ---------------------------------------------------------
        r_p = a * (1 - e**2)
        delta_p = np.maximum(r_p - 1.0 - param, 0.001)
        
        col_name_p = f'vis_p_{param}'
        orb[col_name_p] = (5 * np.log10(np.maximum(r_p, 0.001) * delta_p) + H).astype(float)

        # ---------------------------------------------------------
        # VARIATION 3: Flux-Weighted Mean Distance (The "Photon Count")
        # r_eff = a * (1 - e^2)^(1/4)
        # This represents the distance corresponding to the time-averaged flux.
        # ---------------------------------------------------------
        r_flux = a * np.power(1 - e**2, 0.25)
        delta_flux = np.maximum(r_flux - 1.0 - param, 0.001)
        
        col_name_flux = f'vis_flux_{param}'
        orb[col_name_flux] = (5 * np.log10(np.maximum(r_flux, 0.001) * delta_flux) + H).astype(float)

        # ---------------------------------------------------------
        # 4. ECLIPTIC CROSSING BRIGHTNESS (vis_node)
        # The object crosses the ecliptic at True Anomaly (f) = -omega and 180-omega.
        # We find the radius r at these two points and pick the brighter one (smaller r).
        # ---------------------------------------------------------
        # Polar equation: r = a(1-e^2) / (1 + e*cos(f))
        p_semi = a * (1 - e**2)
        
        # Node 1: f = -omega (Ascending Node usually, depends on convention)
        # cos(-x) = cos(x), so we just use cos(peri_rad)
        r_node1 = p_semi / (1 + e * np.cos(peri_rad))
        
        # Node 2: f = 180 - omega (Descending Node)
        # cos(180 - x) = -cos(x)
        r_node2 = p_semi / (1 - e * np.cos(peri_rad))
        
        # Take the minimum r (brightest node)
        r_node = np.minimum(r_node1, r_node2)
        
        # Calculate Mag
        delta_node = np.maximum(r_node - 1.0 - param, 0.001)
        col_node = f'vis_node_{param}'
        orb[col_node] = (5 * np.log10(np.maximum(r_node, 0.001) * delta_node) + H).astype(float)

    add_advanced_vis_features(orb, param=0.6) 
    add_advanced_vis_features(orb, param=0.0)
    add_advanced_vis_features(orb, param=0.3)

    # 1. Declination of Perihelion (dec_q)
    # Surveys (Pan-STARRS, CSS) are Northern Hemisphere biased. 
    # Objects with perihelion deep in the south are detected less frequently.
    eps = np.radians(23.44)

    # 2. Convert dataframe columns from degrees to radians for calculation
    i_rad = np.radians(orb['i'])
    node_rad = np.radians(orb['Node'])    # Longitude of Ascending Node (Omega)
    peri_rad = np.radians(orb['Peri'])    # Argument of Perihelion (omega)

    # 3. Calculate the Sine of the Declination
    # Formula: sin(dec) = sin(i)sin(w)cos(eps) + [cos(w)sin(Node) + sin(w)cos(i)cos(Node)]sin(eps)
    sin_dec = (
        (np.sin(i_rad) * np.sin(peri_rad) * np.cos(eps)) + 
        ((np.cos(peri_rad) * np.sin(node_rad) + np.sin(peri_rad) * np.cos(i_rad) * np.cos(node_rad)) * np.sin(eps))
    )

    # 4. Calculate final Declination in degrees (arcsin)
    orb['dec_perihelion'] = np.degrees(np.arcsin(sin_dec))

feature_engineering(orb)


# Data Quality, Comparison with Other Databases

In [6]:
# Bring in AstDyS and JPL orbits as a check on the orbital elements quality

# AstDyS SECTION
astdys_names=["Astdys Name","Epoch-MJD","a","e","i","Node","Peri","M","H","G","rand"]
astdys_widths=[15,12,25,25,25,25,25,25,6,6,3]
astdys = pd.read_fwf("https://newton.spacedys.com/~astdys2/catalogs/ufitobs.cat",index_col=False, names=astdys_names, widths=astdys_widths, skiprows=6)
astdys["Astdys Multiopp"] = 1
astdys_sing = pd.read_fwf("https://newton.spacedys.com/~astdys2/catalogs/singopp.cat",index_col=False, names=astdys_names, widths=astdys_widths, skiprows=6)
astdys_sing["Astdys Multiopp"] = 0
# union all these
astdys = pd.concat([astdys, astdys_sing])
print(len(astdys))
astdys = astdys.convert_dtypes()
astdys["n"] = 360/(astdys["a"])**1.5/365.2569
astdys["Perihelion_dist"] = astdys["a"] * (1-astdys["e"])
astdys["Aphelion_dist"] = astdys["a"] * (1+astdys["e"])
astdys["Epoch"] = astdys["Epoch-MJD"]+2400000.5
astdys["Astdys Name"] = astdys["Astdys Name"].str.replace("'","")
astdys["Astdys Name"] = astdys["Astdys Name"].str.slice(0,4)+" "+astdys["Astdys Name"].str.slice(4)
astdys["Ref"] = "AstDyS"
astdys
orb = orb.merge(astdys[["Astdys Name","a","e","i","H","Astdys Multiopp"]], how="left", left_on="Principal_desig", right_on="Astdys Name", suffixes=("", "_astdys"))

# Figure out difference between H and H_astdys
orb["H_diff_abs"] = (orb["H"] - orb["H_astdys"]).abs()
orb["a_diff_abs"] = (orb["a"] - orb["a_astdys"]).abs()
orb["e_diff_abs"] = (orb["e"] - orb["e_astdys"]).abs()
orb["i_diff_abs"] = (orb["i"] - orb["i_astdys"]).abs()
orb["H_diff_abs"] = orb["H_diff_abs"].fillna(0.011)
orb["a_diff_abs"] = orb["a_diff_abs"].fillna(0)
orb["e_diff_abs"] = orb["e_diff_abs"].fillna(0)
orb["i_diff_abs"] = orb["i_diff_abs"].fillna(0)

# set this to 1 any time Astdys Multiopp is defined and disagrees with Num_opps
mpc_multiopp = (orb["Num_opps"] > 1).astype(int)
orb["multi_opp_disagree"] = ((orb["Astdys Multiopp"].notna()) & (orb["Astdys Multiopp"] != mpc_multiopp)).astype(int)


# JPL SECTION
jpl_names=["Desig","Epoch-MJD","a","e","i","Peri","Node","M","H","G","Ref"]
jpl_widths=[14,6,12,11,10,10,10,12,6,5,10]
jpl = pd.read_fwf("https://ssd.jpl.nasa.gov/dat/ELEMENTS.UNNUM.gz",compression='gzip', index_col=False, names=jpl_names, widths=jpl_widths, skiprows=2)
jpl = jpl.convert_dtypes()

# for JPL, we will only compare H for now as that's the thing we most need to be certain of
orb = orb.merge(jpl[["Desig","H"]], how="left", left_on="Principal_desig", right_on="Desig", suffixes=("", "_jpl"))
orb["H_diff_abs_jpl"] = (orb["H"] - orb["H_jpl"]).abs()
orb["H_diff_abs_jpl"] = orb["H_diff_abs_jpl"].fillna(0.012)

orb["H_diff_abs_max"] = orb[["H_diff_abs","H_diff_abs_jpl"]].max(axis=1)

# # Filter out situations where the absolute magnitude disagrees between MPC and AstDyS or JPL
# orb = orb[orb["H_diff_abs_max"] < 3]

# Indicator variable, the training target for binary classification model
orb["Is_Past_Threshold"] = (orb["Num_opps"] >= NUM_OPPS_FOR_TRAINING)*1

orb["Num_opps_minus_one"] = orb["Num_opps"] - 1

551745


In [7]:
# Mislinkage filtering by bringing in a training dataset and training a probabilistic binary classifier
mislinkage_training = pd.read_csv("./mislinkage classifier/mislinkage training.csv")
mislinkage_training = mislinkage_training[["Num_obs", "rms", "Arc_length", "likely_or_possible_mislinkage"]]
X_df_misl = mislinkage_training.drop(columns=["likely_or_possible_mislinkage"])
y_df_misl = mislinkage_training["likely_or_possible_mislinkage"]
xgb_misl = XGBClassifier()
xgb_misl.fit(X_df_misl, y_df_misl)
orb["mislinkage_pred"] = xgb_misl.predict_proba(orb[X_df_misl.columns])[:,1]

In [8]:
# Named filter lists

# This is NOT used in any way to bias the model results, and if you remove it, the model will still behave overall in roughly the same way.
# Excluding them may improve model performance by reducing noise in the training data, and may allow the next layer of discoveries to emerge more clearly.
known_strongly_suspected_active_objects = ['2001 BV70',
 '2019 OE31',
 '2008 BJ22',
 '2007 VB146',
 '2025 HV38',
 '2015 BC566',
 '2025 VZ8',
 '2021 AY8',
 '2009 FP8',
 '2018 BJ11',
 '2002 CW116']

# MPC lists these as (2060) for example and pandas makes that a negative number. We are removing dual asteroid comet designations from the training set.
dual_designation_list = [-2060, -4015, -7968, -60558, -118401, -300163, -323137, -457175, -523622]

In [9]:
# A variety of filters to select decent quality orbits for training. We want as few as possible poorly defined orbits and mislinkages in the training database.
# In some ways less is more at this stage so that the model is not confused by noisy data.

orb_decent_orbit = orb[((orb["Arc_length"]>=MIN_ARC_LENGTH_FOR_TRAINING)|(orb["Arc_length"].isna()))
                       &(orb["Num_obs"]>=MIN_NUM_OBS_FOR_TRAINING)
                       # the following 5 filters are to further filter out situations of disagreement between MPC and AstDyS
                       &(orb["H_diff_abs_max"]<0.3)
                       &(orb["a_diff_abs"]<0.0005)
                       &(orb["e_diff_abs"]<0.00015)
                       &(orb["i_diff_abs"]<0.003)
                       &(orb["multi_opp_disagree"]==0)
                       # filter out objects with some probability of being mislinked
                       &(orb["mislinkage_pred"]<0.03)
                       # Use MPC U parameter as another filter
                       &(orb["U"]<9)
                       # We will not train on known active objects as the model is supposed to reflect the inert asteroid population
                       &~orb["Principal_desig"].isin(known_strongly_suspected_active_objects)
                       &~orb["Number"].isin(dual_designation_list)]

print(len(orb_decent_orbit))

orb_training = orb_decent_orbit


1339464


In [10]:
# These are the features we will use for classification. Ignore all the others in the data. These were selected based on model performance.
mlcols = ['H', 'Node', 'i', 'Tp', 'vis_0', 'visq_0', 'orbital_period_sync', 'gal_dist', 'Perihelion_direction_x_e', 'Perihelion_direction_y_e', 'Perihelion_direction_z_e', 'vis_Q_0.6', 'vis_p_0.6', 'vis_flux_0.0', 'vis_flux_0.3', 'TJ', 'dec_perihelion', 'Is_Past_Threshold']

# for the regression model, swap "Is_Past_Threshold" with "Num_opps" in the feature list
mlcols_reg = mlcols.copy()
mlcols_reg.remove("Is_Past_Threshold")
mlcols_reg.append("Num_opps_minus_one")

data_df = orb_training.dropna(subset=["H"])[mlcols].astype(np.float32)
data_df_reg = orb_training.dropna(subset=["H"])[mlcols_reg].astype(np.float32)

In [11]:
# hyperparameters
hyperparam = {
    "GBM": [
        {},
        {
            "learning_rate": 0.03,
            "num_leaves": 128,
            "feature_fraction": 0.9,
            "min_data_in_leaf": 3,
            "ag_args": {"name_suffix": "Large", "priority": 0, "hyperparameter_tune_kwargs": None},
        },
    ],
    "XGB": [{}, {"learning_rate": 0.5, "max_depth": 3, "min_child_weight": 18, "subsample": 1.0, "ag_args": {"name_suffix": "_tuned"}}]
}

hyperparam_poisson = {
    'GBM': {'objective': 'poisson'},
    'XGB': {'objective': 'count:poisson'},
    'CAT': {'objective': 'Poisson'}
}

hyperparam_tweedie = {
    'GBM': { # LightGBM
        'objective': 'tweedie',
        'tweedie_variance_power': 1.5, # Range (1, 2)
    },
    'XGB': { # XGBoost
        'objective': 'reg:tweedie',
        'tweedie_variance_power': 1.5, # Range (1, 2)
    },
    'CAT': { # CatBoost
        'loss_function': 'Tweedie:variance_power=1.5',
    }
}

In [12]:
from sklearn.metrics import mean_poisson_deviance, mean_tweedie_deviance
from autogluon.core.metrics import make_scorer

# Create the Poisson Scorer
ag_poisson_scorer = make_scorer(
    name='mean_poisson_deviance',
    score_func=mean_poisson_deviance,
    optimum=0,
    greater_is_better=False
)

# IN FUTURE, we will evaluate Tweedie Regression vs. Poisson Regression
# Create the Tweedie Scorer (requires power parameter which should be tuned)
ag_tweedie_scorer = make_scorer(
    name='mean_tweedie_deviance',
    score_func=mean_tweedie_deviance,
    optimum=0,
    greater_is_better=False,
    power=1.5  # You can adjust this based on your data distribution
)

In [ ]:
# KEY TO REGRESSION: We see the first opposition as a given. No object ends up in any orbit database with zero oppositions.
# Then each additional opposition is an approximately Poisson process dependent on visibility factors.
# Poisson assumptions are indeed violated somewhat (survey depth not constant forever in time/history) but this is still likely the best model to use—
# except possibly Tweedie regression which we will evaluate in future work.

# Regression model training
predictor_reg = TabularPredictor(label="Num_opps_minus_one", eval_metric=ag_poisson_scorer, problem_type='regression')
# Note that in some environments we are seeing problems with parallel fitting using Ray, so sequential is safer and yields more consistent results across more machines and more environments.
predictor_reg.fit(data_df_reg, presets="good_quality", hyperparameters=hyperparam_poisson, num_stack_levels=2, dynamic_stacking=False, ag_args_ensemble={"fold_fitting_strategy": "sequential_local"}, time_limit=TRAINING_TIME_LIMIT/2)

# make out-of-fold predictions and add them to orb as well as to data_df
# Ultimately this allows the binary classifier model to leverage the regression model's predictions as a feature
data_df["exp_Num_opps"] = predictor_reg.predict(data_df) + 1
data_df["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)

No path specified. Models will be saved in: "AutogluonModels/ag-20260204_130958"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.5.0: Tue Apr 22 19:54:49 PDT 2025; root:xnu-11417.121.6~2/RELEASE_ARM64_T6000
CPU Count:          10
Memory Avail:       5.77 GB / 32.00 GB (18.0%)
Disk Space Avail:   42.56 GB / 926.35 GB (4.6%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.


In [ ]:
# Binary Classifier Training
predictor = TabularPredictor(label="Is_Past_Threshold", eval_metric='log_loss')
# Note that in some environments we are seeing problems with parallel fitting using Ray, so sequential is safer and yields more consistent results across more machines and more environments.
predictor.fit(data_df, presets="good_quality", hyperparameters=hyperparam, num_stack_levels=2, dynamic_stacking=False, ag_args_ensemble={"fold_fitting_strategy": "sequential_local"}, time_limit=TRAINING_TIME_LIMIT/2)


No path specified. Models will be saved in: "AutogluonModels/ag-20260204_121135"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.5.0: Tue Apr 22 19:54:49 PDT 2025; root:xnu-11417.121.6~2/RELEASE_ARM64_T6000
CPU Count:          10
Memory Avail:       7.72 GB / 32.00 GB (24.1%)
Disk Space Avail:   36.67 GB / 926.35 GB (4.0%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
Beginning AutoGluon training ... Time limit = 250s
AutoGluon will save models to "/Users/petervw/Git

In [ ]:
# This is interesting as it lets you see which models were used in the stack/ensemble and how they performed in cross-validation
# Regression leaderboard
predictor_reg.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L4,-0.398253,mean_poisson_deviance,6.329864,172.477015,0.015442,1.120162,4,False,6
1,LightGBM_BAG_L2,-0.398322,mean_poisson_deviance,6.314422,171.356853,3.356653,86.238006,2,False,3
2,WeightedEnsemble_L3,-0.398322,mean_poisson_deviance,6.331558,171.402543,0.017136,0.045690,3,False,4
3,XGBoost_BAG_L1,-0.399122,mean_poisson_deviance,2.957769,85.118847,2.957769,85.118847,1,False,1
4,WeightedEnsemble_L2,-0.399122,mean_poisson_deviance,2.973478,85.159705,0.015709,0.040858,2,False,2
5,LightGBM_BAG_L3,-0.401489,mean_poisson_deviance,7.614590,213.976522,1.300168,42.619669,3,False,5
6,XGBoost_BAG_L1_FULL,NaN,mean_poisson_deviance,NaN,10.242034,NaN,10.242034,1,True,7
7,WeightedEnsemble_L4_FULL,NaN,mean_poisson_deviance,NaN,19.749402,NaN,1.120162,4,True,12
8,WeightedEnsemble_L3_FULL,NaN,mean_poisson_deviance,NaN,18.674930,NaN,0.045690,3,True,10
9,WeightedEnsemble_L2_FULL,NaN,mean_poisson_deviance,NaN,10.282892,NaN,0.040858,2,True,8


In [ ]:
# Same thing for the binary classifier
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L4,-0.157041,log_loss,7.239435,211.835137,0.028942,2.162199,4,False,8
1,WeightedEnsemble_L2,-0.157065,log_loss,3.925734,101.325959,0.029164,0.830922,2,False,3
2,XGBoost_BAG_L1,-0.157082,log_loss,2.180531,56.576909,2.180531,56.576909,1,False,2
3,WeightedEnsemble_L3,-0.157117,log_loss,6.054731,168.919172,0.029804,0.917929,3,False,6
4,LightGBM_BAG_L2,-0.157140,log_loss,5.079983,140.401106,1.183413,39.906069,2,False,4
5,XGBoost_BAG_L2,-0.157182,log_loss,4.841514,128.095174,0.944944,27.600137,2,False,5
6,LightGBM_BAG_L3,-0.157235,log_loss,7.210493,209.672938,1.185565,41.671695,3,False,7
7,LightGBM_BAG_L1,-0.158201,log_loss,1.716039,43.918128,1.716039,43.918128,1,False,1
8,XGBoost_BAG_L2_FULL,NaN,log_loss,NaN,13.996945,NaN,3.048945,2,True,13
9,XGBoost_BAG_L1_FULL,NaN,log_loss,NaN,6.716487,NaN,6.716487,1,True,10


In [ ]:
# Anything with more than 16 oppositions doesn't need much analysis, and no absolute magnitude means fundamentally impossible to apply these methods
orb_pred = orb[(orb["Num_opps"]<=16)].dropna(subset=["H_MPC","H_astdys","H_jpl"], how='all')

# For prediction purposes we will assume that the highest of the three H values is the correct one and the others are erroneously bright
orb_pred["H"] = orb_pred[["H_MPC","H_astdys","H_jpl"]].max(axis=1)

# Now the predictions for the regression model
orb_pred["exp_Num_opps"] = predictor_reg.predict(orb_pred) + 1
orb_pred["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)

# Predictions for the probabalistic binary classification model
# predict_proba to set probs in orb_pred (this sets the probability for anything not in the training dataset, but anything in the training will get overridden in the next line)
orb_pred["prob"] = predictor.predict_proba(orb_pred)[1]
# override the prob field in orb_pred to ag_probs lining up data index, but only for the rows that exist in both (same index in both)
orb_pred["prob"].update(TabularPredictor.predict_proba_oof(predictor)[1])

orb_pred["Num_opps_diff"] = orb_pred["exp_Num_opps"] - orb_pred["Num_opps"]
orb_pred["Num_opps_mult"] = orb_pred["exp_Num_opps"] / orb_pred["Num_opps"]

Using OOF from "WeightedEnsemble_L4" as a proxy for "WeightedEnsemble_L4_FULL".
/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_45126/1297657982.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  orb_pred["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)
Using OOF from "WeightedEnsemble_L4" as a proxy for "WeightedEnsemble_L4_FULL".
/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_45126/1297657982.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series th

In [ ]:
final = orb_pred.copy().sort_values("prob",ascending=False)
# renames of Perihelion_dist to q and Aphelion_dist to Q for brevity
final.rename(columns={"Perihelion_dist":"q","Aphelion_dist":"Q"},inplace=True)

final.set_index("Principal_desig", inplace=True)

# Predictions, Starting with Asteroid Orbits
All predictions here are out-of-fold predictions, that is each prediction is made using training data that excludes itself.
This is possible because AutoGluon uses cross validation and we are using the predict_proba_oof method for data in the training dataset.

In [ ]:
# Single-opposition asteroid-only list (T_J > 3.02), sorted by prob
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["Arc_length"]>11)
      &(final["Num_obs"]>7)
      &(final["prob"]>0.997)
      &(final["TJ"]>3.02)
      &(final["mislinkage_pred"]<0.07)
      &(final["H_diff_abs_max"]<0.6)
      &(final["Orbital_period"]<200)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2008 BJ22,53,MPO732621,83,4,3.072986,0.042267,11.51171,15.05,2.9431,3.202873,3.19791,0.67,0.24,0.999983,20.219362,19.219362,20.219362,0.000654
2025 HV38,39,MPO914081,27,9,2.490837,0.047114,7.23675,15.79,2.373483,2.608191,3.460116,0.92,0.04,0.999983,19.889820,18.88982,19.88982,0.000239
2009 US21,12,MPO190709,24,10,2.611654,0.289262,8.19199,16.31,1.856203,3.367105,3.334779,0.39,0.011,0.999983,16.873455,15.873455,16.873455,0.029863
2025 VZ8,48,E2026-BC2,29,7,2.620987,0.085922,12.09945,16.73,2.395785,2.846189,3.367964,0.66,0.09,0.999983,12.807384,11.807384,12.807384,0.001060
2015 BC566,52,MPO771428,39,5,3.065828,0.03653,11.68739,16.33,2.953833,3.177822,3.199504,0.42,0.0,0.999982,13.549860,12.54986,13.54986,0.000112
2002 JG148,9,MPO 48238,20,10,2.499089,0.126938,10.69329,17.1,2.18186,2.816317,3.432969,0.62,0.011,0.999981,12.542549,11.542549,12.542549,0.007117
2007 VB146,13,MPO130330,16,10,2.752892,0.113703,3.49271,16.94,2.439879,3.065905,3.332674,0.38,0.04,0.999981,14.594437,13.594437,14.594437,0.011756
2016 CV88,10,MPO900357,51,6,2.94445,0.060942,12.23849,16.69,2.765008,3.123892,3.234673,0.9,0.0,0.999979,11.627445,10.627445,11.627445,0.014163
2022 UD92,12,MPO727159,27,7,2.932261,0.403429,15.27327,16.21,1.749302,4.11522,3.099698,0.99,0.011,0.999978,12.039207,11.039207,12.039207,0.003190


In [ ]:
# Same thing, but different sort
# Single-opposition asteroid-only list (T_J > 3.02), sorted by Expeected Number of Oppositions
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["Arc_length"]>11)
      &(final["Num_obs"]>7)
      &(final["prob"]>0.99)
      &(final["TJ"]>3.02)
      &(final["mislinkage_pred"]<0.15)
      &(final["H_diff_abs_max"]<0.6)
      &(final["Orbital_period"]<20)].sort_values("exp_Num_opps", ascending=False)
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2008 BJ22,53,MPO732621,83,4,3.072986,0.042267,11.51171,15.05,2.9431,3.202873,3.19791,0.67,0.24,0.999983,20.219362,19.219362,20.219362,0.000654
2025 HV38,39,MPO914081,27,9,2.490837,0.047114,7.23675,15.79,2.373483,2.608191,3.460116,0.92,0.04,0.999983,19.889820,18.88982,19.88982,0.000239
2009 VW92,12,MPO172380,22,10,2.240262,0.202417,8.57842,16.6,1.786795,2.693728,3.59331,0.68,0.09,0.999983,18.109953,17.109953,18.109953,0.076454
2009 US21,12,MPO190709,24,10,2.611654,0.289262,8.19199,16.31,1.856203,3.367105,3.334779,0.39,0.011,0.999983,16.873455,15.873455,16.873455,0.029863
2017 VJ70,9,MPO773226,24,8,3.042474,0.223791,12.59677,15.4,2.361594,3.723353,3.164836,0.38,0.011,0.999983,16.125671,15.125671,16.125671,0.106863
2007 VB146,13,MPO130330,16,10,2.752892,0.113703,3.49271,16.94,2.439879,3.065905,3.332674,0.38,0.04,0.999981,14.594437,13.594437,14.594437,0.011756
2015 BC566,52,MPO771428,39,5,3.065828,0.03653,11.68739,16.33,2.953833,3.177822,3.199504,0.42,0.0,0.999982,13.549860,12.54986,13.54986,0.000112
2022 TD25,13,MPO718822,28,6,3.043005,0.100492,0.37131,17.1,2.737209,3.348802,3.231566,1.12,0.13,0.999975,13.180348,12.180348,13.180348,0.001140
2010 JK165,11,MPO176952,36,10,2.99782,0.128815,17.12713,16.0,2.611655,3.383985,3.174304,0.35,0.16,0.999981,13.064990,12.06499,13.06499,0.114435


In [ ]:
# Multiple-opp MBC candidates. Currently none look promising and (606367) 2017 VB is likely underobserved simply
# because it is so far north at each opposition given orbital period of about 1.99 years.
cols_to_show = ["Num_obs","Num_opps","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["a"].between(1,5))
      &(final["Orbital_period"]<30)
      &(final["Num_obs"]>20)
      &(final["Num_opps"]>2)
      &(final["prob"]>0.9997)
      &(final["TJ"]>=3)
      &(final["Num_opps_mult"]>2.5)].sort_values("Num_opps_mult", ascending=False)
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Num_opps,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,,
2007 VD199,27,3,MPO955436,<NA>,2,2.595797,0.062804,4.93574,17.61,2.43277,2.758824,3.40904,0.54,0.0,0.999979,12.061371,9.061371,4.020457,0.000702
2014 HT351,27,3,MPO791423,<NA>,4,3.049252,0.030706,2.61342,17.16,2.955623,3.142882,3.235093,0.22,0.15,0.999915,11.098300,8.0983,3.699433,0.000636
2015 KT252,31,3,E2023-XF2,<NA>,3,2.657624,0.094357,4.17288,17.67,2.406859,2.908389,3.377001,0.28,0.31,0.999940,11.014531,8.014531,3.67151,0.000656
2020 RZ11,30,3,E2020-UQ6,<NA>,4,2.904251,0.078661,3.18713,17.61,2.675799,3.132703,3.278817,0.4,0.21,0.999872,10.566649,7.566649,3.522216,0.000256
2021 XB13,37,3,E2026-AB9,<NA>,1,2.531042,0.22409,4.61271,17.86,1.963861,3.098222,3.410728,1.1,0.08,0.999903,10.008548,7.008548,3.336183,0.000172
2012 TN346,48,3,MPO517651,<NA>,1,2.322853,0.173454,7.10866,18.26,1.919944,2.725763,3.545876,0.18,0.26,0.999897,9.798630,6.79863,3.26621,0.001127
2017 PS75,34,3,MPO716458,<NA>,2,3.070161,0.065468,16.4273,17.06,2.869164,3.271159,3.165151,0.3,0.12,0.999725,9.013327,6.013327,3.004442,0.002294
2025 TK16,41,4,E2025-W84,<NA>,3,1.589584,0.618437,7.35617,19.3,0.606526,2.572641,4.134746,0.54,0.05,0.999896,11.461581,7.461581,2.865395,0.000702
2015 YN11,67,4,MPO893723,<NA>,2,2.708469,0.071314,3.71709,17.71,2.515318,2.90162,3.357305,0.5,0.06,0.999962,11.115785,7.115785,2.778946,0.000901


# On to Comet Orbits

In [ ]:
# Multiple-opp Comets
cols_to_show = ["Num_obs","Num_opps","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(~final["a"].between(5,5.4))
      &(final["Orbital_period"]<30)
      &(final["Num_obs"]>14)
      &(final["Num_opps"]>1)
      &(final["prob"]>0.97)
      &(final["TJ"]<=3.02)
      &(final["Num_opps_mult"]>2.5)].sort_values("Num_opps_mult", ascending=False)
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Num_opps,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,,
2010 RH69,31,3,MPO619842,<NA>,5,4.532514,0.056522,11.54792,13.95,4.276327,4.788701,2.97391,0.96,0.19,0.999981,18.775791,15.775791,6.258597,0.000301
2008 GO98,1549,7,E2025-XA4,<NA>,0,3.97198,0.27862,15.55684,12.91,2.865306,5.078655,2.926703,0.88,0.012,0.999981,23.775238,16.775238,3.396463,0.000309
2017 QN84,213,4,MPO907046,<NA>,0,3.771028,0.34354,12.07174,15.1,2.47553,5.066526,2.943421,1.13,0.13,0.999982,12.715179,8.715179,3.178795,0.000154
2003 BM80,226,7,MPO953186,<NA>,0,4.234979,0.187734,5.81293,13.62,3.439928,5.03003,2.991765,1.23,0.012,0.999968,21.722540,14.72254,3.10322,0.000225
2022 AP7,42,3,E2024-EF2,<NA>,1,2.922668,0.715216,13.8422,17.36,0.832329,5.013006,2.797433,0.31,0.03,0.987156,8.487459,5.487459,2.829153,0.004351


In [ ]:
# Single-opp Periodic Comets (excluding Trojans), period < 200 years
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(~final["a"].between(5,5.4))
      &(final["Orbital_period"]<200)
      &(final["Arc_length"]>14)
      &(final["Num_obs"]>7)
      &(final["TJ"]<=3.02)
      &(final["prob"]>0.8)
      &(final["Tp"]>2450000)
      &(final["mislinkage_pred"]<0.15)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2001 BV70,32,MPO 13809,50,10,3.593632,0.425907,4.33032,15.14,2.063079,5.124185,2.947403,0.58,0.44,0.999983,19.126778,18.126778,19.126778,0.000451
2019 OE31,32,E2024-JA9,137,3,4.380092,0.100133,5.22173,14.66,3.941499,4.818685,3.006111,0.65,0.02,0.999981,16.488960,15.48896,16.48896,0.000654
2018 BJ11,46,MPO750185,107,5,4.1806,0.224501,3.42476,15.55,3.242051,5.11915,2.988437,0.47,0.0,0.999467,10.331442,9.331442,10.331442,0.000498
2020 QQ62,18,MPO605265,61,4,3.930811,0.21286,3.78548,16.68,3.094097,4.767525,3.018478,0.17,0.18,0.982886,6.166741,5.166741,6.166741,0.007365
2020 SL9,36,MPO600171,63,3,3.282766,0.237991,23.15256,16.77,2.501498,4.064034,3.003659,0.22,0.37,0.974793,7.653072,6.653072,7.653072,0.000690
2024 XE22,22,MPO929986,32,5,5.86087,0.226553,27.15535,14.24,4.533074,7.188666,2.727344,0.47,0.07,0.970979,5.750088,4.750088,5.750088,0.001992
2012 XK149,12,MPO602393,19,6,3.953996,0.327059,4.02095,16.5,2.660806,5.247185,2.95944,0.21,0.011,0.970160,5.824835,4.824835,5.824835,0.026700
2023 UY75,9,MPO811376,32,8,3.371593,0.480983,17.77052,16.54,1.749914,4.993273,2.887357,0.48,0.12,0.949121,5.529912,4.529912,5.529912,0.044654
2017 FV167,18,MPO523185,32,5,3.164992,0.259195,26.02274,17.27,2.344642,3.985342,2.997751,0.27,0.27,0.933969,5.211908,4.211908,5.211908,0.079845


In [ ]:
# Trojans (2019 GX103 and 2021 UA56 are interesting but still likely sub threshold, a good ITF search should turn them up?)
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["a"].between(5,5.4))
      &(final["Orbital_period"]<200)
      &(final["Arc_length"]>14)
      &(final["Num_obs"]>7)
      &(final["prob"]>0.98)
      &(final["Tp"]>2450000)
      &(final["mislinkage_pred"]<0.15)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2019 GX103,14,MPO621431,54,6,5.277611,0.023604,17.96843,14.89,5.153037,5.402185,2.901374,0.1,0.29,0.997546,8.405168,7.405168,8.405168,0.001229
2014 ER176,14,MPO473314,25,10,5.105578,0.122835,9.92579,14.97,4.478433,5.732723,2.955835,0.28,0.17,0.997427,8.240490,7.24049,8.24049,0.013499
2021 UA56,15,MPO666614,28,5,5.296787,0.044958,14.17541,14.8,5.058654,5.534921,2.936815,0.19,0.011,0.995858,8.276093,7.276093,8.276093,0.004683
2025 QU66,11,MPO950580,31,6,5.20243,0.03932,19.55756,14.84,4.997872,5.406989,2.88316,0.9,0.16,0.989036,7.161485,6.161485,7.161485,0.094406
2021 SG44,9,MPO666061,40,4,5.275842,0.035844,25.58829,14.7,5.086735,5.464949,2.801451,0.07,0.011,0.988681,7.139878,6.139878,7.139878,0.040482
2024 SE60,14,MPO872967,34,6,5.311456,0.036887,2.14312,15.01,5.115531,5.507381,2.99753,0.91,0.011,0.987665,7.512152,6.512152,7.512152,0.006479
2021 VL33,10,MPO666802,25,5,5.199192,0.053534,9.08346,15.1,4.920859,5.477525,2.972097,0.08,0.011,0.985254,7.066749,6.066749,7.066749,0.073831
2024 YL57,11,MPO900801,65,5,5.184668,0.097898,5.64024,15.29,4.6771,5.692236,2.9808,1.08,0.15,0.981055,6.927179,5.927179,6.927179,0.038699


In [ ]:
# Single-opp Comets, period > 200 years
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(~final["a"].between(5,5.4))
      &(final["Orbital_period"]>=200)
      &(final["Arc_length"]>14)
      &(final["Num_obs"]>11)
      &(final["TJ"]<=3.02)
      &(final["prob"]>0.6)
      &(final["Tp"]>2450000)
      &(final["mislinkage_pred"]<0.5)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2000 KP65,20,MPO 50477,58,2,87.730516,0.962767,45.52217,11.04,3.266479,172.194553,1.6148,0.61,0.54,0.987055,8.552525,7.552525,8.552525,0.000898
2005 VX3,52,MPO 97960,81,10,975.748422,0.995754,112.6904,14.12,4.14332,1947.353524,-0.967277,0.52,0.02,0.771374,3.965892,2.965892,3.965892,0.001230
2023 DP5,47,E2024-U35,134,4,91.58374,0.947553,118.35367,14.5,4.80332,178.36416,-1.216786,0.88,0.0,0.617742,3.689335,2.689335,3.689335,0.000309


In [ ]:
# JFC / Centaur List (see https://arxiv.org/pdf/2512.11204)
cols_to_show = ["Num_obs","Num_opps","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[((final["Arc_length"]>11)| (final["Arc_length"].isna()))
      &(final["Num_obs"]>7)
      &(final["mislinkage_pred"]<0.5)
      &(final["a"].between(4.05,5.05))
      &(final["TJ"]<=3.05)
      &(final["Orbital_period"]<200)].sort_values("Num_opps_mult", ascending=False)
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Num_opps,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,,
2019 OE31,32,1,E2024-JA9,137,3,4.380092,0.100133,5.22173,14.66,3.941499,4.818685,3.006111,0.65,0.02,0.999981,16.488960,15.48896,16.48896,0.000654
2018 BJ11,46,1,MPO750185,107,5,4.1806,0.224501,3.42476,15.55,3.242051,5.11915,2.988437,0.47,0.0,0.999467,10.331442,9.331442,10.331442,0.000498
2015 WE43,10,1,MPO885769,15,9,4.912133,0.059632,9.49817,15.02,4.61921,5.205056,2.972455,0.44,0.011,0.996335,7.987283,6.987283,7.987283,0.167028
2010 RH69,31,3,MPO619842,<NA>,5,4.532514,0.056522,11.54792,13.95,4.276327,4.788701,2.97391,0.96,0.19,0.999981,18.775791,15.775791,6.258597,0.000301
2013 AP182,14,2,MPO660785,<NA>,4,4.779508,0.374829,24.41077,14.25,2.988008,6.571008,2.706868,0.48,0.45,0.999970,10.463051,8.463051,5.231525,0.009337
2025 QH62,13,1,MPO950566,23,9,4.301469,0.636862,13.95404,16.85,1.562025,7.040913,2.570229,0.24,0.0,0.442331,3.795718,2.795718,3.795718,0.120087
2024 VW18,16,1,MPO907397,67,4,4.149558,0.293912,14.81622,16.85,2.929955,5.369161,2.904311,0.94,0.16,0.639394,3.781180,2.78118,3.78118,0.012492
2008 SW117,18,1,MPO147501,34,10,4.423787,0.37039,4.29903,16.64,2.785259,6.062316,2.884324,0.45,0.04,0.518808,3.267963,2.267963,3.267963,0.003454
2024 BA29,13,1,E2024-RG0,97,4,4.504982,0.217059,14.96547,16.42,3.527136,5.482828,2.909971,1.11,0.16,0.295461,3.160488,2.160488,3.160488,0.000636


# Possible mislinkages
The possible mislinkage classifier is only for filtering and prioritization but there will be real discoveries to be had among the possible mislinkages!

In [ ]:
# Posible mislinkage list (lower likelihood of being a possible mislinkage), more likely to be real objects than the next list
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["Arc_length"]>11)
      &(final["Num_obs"]>7)
      &(final["prob"]>0.999)
      &( final["mislinkage_pred"].between(0.07,0.6) | (final["H_diff_abs_max"]>0.6) )
      &(final["mislinkage_pred"]<0.6)
      &(final["Orbital_period"]<200)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2017 VJ70,9,MPO773226,24,8,3.042474,0.223791,12.59677,15.4,2.361594,3.723353,3.164836,0.38,0.011,0.999983,16.125671,15.125671,16.125671,0.106863
2014 DZ209,11,MPO869696,22,6,2.442285,0.162407,1.26563,17.17,2.045642,2.838929,3.482115,1.03,0.011,0.999983,16.822491,15.822491,16.822491,0.201786
2009 AN42,9,MPO156162,30,10,2.603329,0.165357,1.34694,16.7,2.172851,3.033807,3.393445,0.44,0.74,0.999983,17.676283,16.676283,17.676283,0.246543
2009 VW92,12,MPO172380,22,10,2.240262,0.202417,8.57842,16.6,1.786795,2.693728,3.59331,0.68,0.09,0.999983,18.109953,17.109953,18.109953,0.076454
2023 WA39,11,MPO793403,20,7,2.701999,0.253494,10.924,16.66,2.017057,3.38694,3.294543,0.4,0.011,0.999982,12.937577,11.937577,12.937577,0.525645
2007 TD441,12,MPO167690,36,10,2.877535,0.173955,10.85057,16.33,2.376974,3.378095,3.246633,0.48,0.03,0.999982,13.481244,12.481244,13.481244,0.304759
2008 XD26,10,MPO172336,18,10,3.013373,0.148006,5.84507,16.61,2.567377,3.459369,3.224099,0.34,0.11,0.999982,13.047234,12.047234,13.047234,0.384012
2010 JK165,11,MPO176952,36,10,2.99782,0.128815,17.12713,16.0,2.611655,3.383985,3.174304,0.35,0.16,0.999981,13.064990,12.06499,13.06499,0.114435
2007 DK94,10,MPO118214,24,10,2.748064,0.30486,7.46267,16.8,1.910288,3.58584,3.265921,0.31,0.05,0.999981,12.692546,11.692546,12.692546,0.263499


In [ ]:
# Higher likelihood of being a possible mislinkage, less likely to be real objects than the prior list
# Useful as an indicator of the very real presence of mislinkages in the orbital database (mostly 3-night linkages, mostly longer arcs than can be fully supported by only 3 nights)
cols_to_show = ["Num_obs","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["Arc_length"]>15)
      &(final["Num_obs"]>5)
      &(final["prob"]>0.9999)
      &(final["mislinkage_pred"]>0.7)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,
2018 AM82,10,MPO870735,25,6,2.314835,0.212282,8.1539,17.16,1.823438,2.806233,3.538115,1.17,0.0,0.999983,14.667699,13.667699,14.667699,0.959044
2017 JY13,7,E2024-O09,26,5,3.080556,0.27611,10.74528,16.36,2.229982,3.931129,3.142147,0.7,0.011,0.999977,11.948117,10.948117,11.948117,0.874018
2020 QB71,10,MPO956153,33,6,2.762551,0.151613,5.31322,17.21,2.343712,3.18139,3.317699,0.99,0.01,0.999975,12.722162,11.722162,12.722162,0.907683
2001 QP301,7,MPO181625,32,10,3.097701,0.189358,9.98591,16.53,2.511128,3.684274,3.17196,0.19,0.03,0.999974,11.154137,10.154137,11.154137,0.791343
2018 RX128,10,MPO870873,29,7,2.456876,0.310781,7.22633,17.52,1.693325,3.220427,3.41364,0.9,0.011,0.999971,10.549479,9.549479,10.549479,0.909235
2023 UN22,10,MPO775864,20,8,2.684576,0.063419,2.75592,17.84,2.514323,2.854829,3.370175,0.2,0.06,0.999934,10.982738,9.982738,10.982738,0.847907
2023 RE178,9,MPO886331,37,5,3.045047,0.383957,19.35603,16.27,1.87588,4.214213,3.041579,1.34,0.16,0.999931,10.103250,9.10325,10.10325,0.836593
2016 RF103,9,MPO839406,20,7,2.818402,0.492374,5.30519,16.69,1.430695,4.20611,3.121789,0.16,0.011,0.999930,11.167073,10.167073,11.167073,0.742961


In [ ]:
# Another list which may largely be mislinkages (2-opp mislinkages are more common than we would like)
cols_to_show = ["Num_obs","Num_opps","Ref","Arc_length","U","a","e","i","H","q","Q","TJ","rms","H_diff_abs_max","prob", "exp_Num_opps", "Num_opps_diff", "Num_opps_mult", "mislinkage_pred"]
tempout = final[(final["Num_opps"]==2)
      &(final["Num_obs"]>5)
      &(final["prob"]>0.999)]
tempout = tempout[cols_to_show]
tempout.head(20)

,Num_obs,Num_opps,Ref,Arc_length,U,a,e,i,H,q,Q,TJ,rms,H_diff_abs_max,prob,exp_Num_opps,Num_opps_diff,Num_opps_mult,mislinkage_pred
Principal_desig,,,,,,,,,,,,,,,,,,,
2017 BX48,16,2,MPO492051,<NA>,4,2.764822,0.150599,5.26496,17.92,2.348444,3.181201,3.317078,0.31,0.42,0.999971,10.497226,8.497226,5.248613,0.521370
2013 AP182,14,2,MPO660785,<NA>,4,4.779508,0.374829,24.41077,14.25,2.988008,6.571008,2.706868,0.48,0.45,0.999970,10.463051,8.463051,5.231525,0.009337
2020 RE97,19,2,MPO605455,<NA>,5,2.803275,0.107939,7.94326,17.73,2.500691,3.10586,3.301497,0.21,0.33,0.999967,10.353281,8.353281,5.176641,0.001197
2020 TB66,18,2,MPO605751,<NA>,5,2.678684,0.089538,4.70628,18.11,2.438839,2.918528,3.366828,0.29,0.41,0.999966,11.190698,9.190698,5.595349,0.003400
2022 BX9,15,2,MPO681541,<NA>,5,2.353895,0.075363,4.9408,19.18,2.176498,2.531291,3.546799,0.34,1.08,0.999957,11.525592,9.525592,5.762796,0.012975
2010 MY6,16,2,MPO602218,<NA>,7,2.784603,0.049106,3.46933,17.77,2.647862,2.921344,3.327182,0.49,0.27,0.999956,11.779732,9.779732,5.889866,0.035262
2013 EH143,18,2,E2024-V74,<NA>,7,2.271064,0.419631,4.25908,17.85,1.318054,3.224074,3.487066,0.25,0.59,0.999947,9.894328,7.894328,4.947164,0.004129
2015 DS183,28,2,MPO678418,<NA>,4,3.092059,0.119953,12.80708,16.74,2.721157,3.462962,3.175281,0.25,0.14,0.999947,10.145393,8.145393,5.072697,0.000563
2014 EQ71,20,2,MPO703687,<NA>,6,2.918478,0.174609,1.62323,17.22,2.408885,3.428072,3.25707,0.87,0.16,0.999937,10.986131,8.986131,5.493065,0.000476


# Import the Comet Orbit File from MPC
Beyond this point we aren't looking for new discoveries but mostly retrospectively looking at older discoveries

In [ ]:
# Objects moved to Comet Orbits
comet = pd.read_json("https://www.minorplanetcenter.net/Extended_Files/allcometels.json.gz",compression='gzip')

comet = comet[comet["Orbit_type"].isin(["P","D","A"])]
# comet = comet[comet["Designation_and_name"].str[0].isin(["P","D"])]

comet["a"] = comet["Perihelion_dist"] / (1 - comet["e"])
comet["Orbital_period"] = np.sqrt(comet["a"]**3)
comet["Tp"] = 2454466 # this is just a dummy for now

feature_engineering(comet)
# Only pulling in the magnitude data for comets from find_orb as the MPC H values in the comet file are not H assuming inert
comet_mags_fo = pd.read_fwf("./comet_orbits/comet_els_fo.txt", widths=[8, 5, 1000], names=["Packed", "Mag", "Discard"])

/Users/petervw/mambaforge/envs/automl5/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
comet["fullpacked"] = comet["Orbit_type"] + comet["Provisional_packed_desig"]
comet.drop(columns=["H"], inplace=True)
cometmerge = comet.merge(comet_mags_fo, left_on="fullpacked", right_on="Packed", how="left")

cometmerge.loc[cometmerge["Designation_and_name"] == "489P/Denning", "Mag"] = 15.35
cometmerge.loc[cometmerge["Designation_and_name"] == "P/2025 UX109 (Ye)", "Mag"] = 15.00

cometmerge["H"] = cometmerge["Mag"]

cometmerge["exp_Num_opps"] = predictor_reg.predict(cometmerge) + 1
cometmerge["prob"] = predictor.predict_proba(cometmerge)[1]
cometmerge["Num_opps"] = pd.NA # no data for this in the comet file

In [ ]:
# This is the 205 rows referenced in the paper which ActivitySCOPE would have found as likely comets if not found morphologically
cometout = cometmerge[(cometmerge["prob"] > 0.8) & ~(cometmerge["H"].isna())].sort_values("prob", ascending=False)
cometout[["Designation_and_name","a","e","i","H",'prob']]#[cometout["fullpacked"].str[:4]=="PK23"]

,Designation_and_name,a,e,i,H,prob
42,P/2004 R3 (LINEAR-NEAT),4.759893,0.255187,12.7856,14.70,0.999985
102,P/2013 G1 (Kowalski),6.898284,0.511029,5.4581,10.19,0.999983
158,P/2017 P1 (PANSTARRS),7.914748,0.310760,7.6886,10.11,0.999983
16,D/1993 F2-H (Shoemaker-Levy),6.825485,0.211776,5.9132,8.47,0.999983
10,D/1993 F2-B (Shoemaker-Levy),6.859515,0.215595,5.9899,10.07,0.999983
...,...,...,...,...,...,...
141,P/2015 X3 (PANSTARRS),5.003588,0.440042,24.4105,14.47,0.855464
201,P/2020 G1 (Pimentel),3.608428,0.860322,18.4996,15.13,0.853429
249,P/2022 L3 (ATLAS),6.502741,0.627715,21.5428,12.85,0.828556
189,A/2019 T1,38.723007,0.889427,120.8769,13.48,0.812869


In [ ]:
# ActivitySCOPE discoveries that currently live in the comet file
discoveries_in_comet_file = ["P/2025 UX109 (Ye)","P/2023 JN16 (Lemmon)","P/2022 BV9 (Lemmon)","P/2017 FL36 (PANSTARRS)","489P/Denning"]
cometmerge[cometmerge["Designation_and_name"].isin(discoveries_in_comet_file)][["Designation_and_name","a","e","i","H",'prob']]

,Designation_and_name,a,e,i,H,prob
154,P/2017 FL36 (PANSTARRS),2.926672,0.032694,15.6870,16.65,0.999982
243,P/2022 BV9 (Lemmon),4.360618,0.234834,11.9339,13.79,0.999983
260,P/2023 JN16 (Lemmon),2.695447,0.146815,3.7026,15.73,0.999983
300,P/2025 UX109 (Ye),3.801405,0.324032,3.1813,15.00,0.999983
891,489P/Denning,4.421537,0.646954,4.0269,15.35,0.988901


# This is the part for the paper
Yes it's less polished code but we may just pull it out before actually publishing it more widely

In [ ]:
# Discoveries, rediscoveries, under investigation and slightly sub threshold examples
objects_to_include_in_data_table = ['2001 BV70',
 '2019 OE31',
 '2008 BJ22',
 '2007 VB146',
 '2025 HV38',
 '2015 BC566',
 '2025 VZ8',
 '2021 AY8',
 '2009 FP8',
 '2018 BJ11',
 '2002 CW116',
 
 '2024 XE22',
 '2009 DP2',
'2020 QQ62',
 
 '2010 RH69','2008 GO98','2017 QN84','2003 BM80',
 
 '2017 VB','2012 XK149','2020 PB54',

 '2021 EA45','2017 PZ20','2020 BO57','2022 PH33', '2010 WE48',
 '2006 PD30',

 
 '2017 FV167','2024 TS82','2014 WL185',]

# False Positives to discuss
objects_fp = ['2025 HB52', # brightest extension in last year
              '2023 UQ87', # photometry issues
              '2025 WW35', '2019 GX101', # one night recoveries
               '2015 FH479', '2014 DZ209','2009 US21','2009 AN42','2009 VW92', #mislinkages
               '2002 GR31','2024 SG49',
              ]

In [ ]:
for_paper = final.copy()
for_paper["Principal_desig"] = for_paper.index
for_paper["Object"] = for_paper["Number"].astype(str) + " " + for_paper["Principal_desig"]
# replace "<NA> " with ""
for_paper["Object"] = for_paper["Object"].str.replace("<NA> ", "")

In [ ]:
statuses = pd.read_excel("Object statuses.xlsx")

In [ ]:
#
# COMET TABLE AND ASTEROID TABLE
#

# 1. Prepare the raw data
# Combine main table and comet table
table_for_paper = pd.concat([
    for_paper[for_paper["Principal_desig"].isin(objects_to_include_in_data_table)][["Object","a","e","i","H","TJ","Num_opps","prob","exp_Num_opps"]],
    cometmerge[cometmerge["Designation_and_name"].isin(discoveries_in_comet_file)][["Designation_and_name","a","e","i","H","TJ","Num_opps","prob","exp_Num_opps"]].rename(columns={"Designation_and_name": "Object"})
], ignore_index=True)

# Merge metadata and sort
table_for_paper = table_for_paper.merge(statuses[["Object","Status"]], on="Object", how="left")
table_for_paper.sort_values(by=["prob"], ascending=False, inplace=True)

# 2. Define column renaming
renames = {
    "TJ": "T_J",
    "prob": "P(N_{\\text{opp}} \ge 4)",
    "Num_opps": "N_{\\text{opp}}",
    "exp_Num_opps": "E[N_{\\text{opp}}]",
    "H": "H_V"
}
table_for_paper = table_for_paper.rename(columns=renames)

# 3. Build the Formatter Dictionary
format_dict = {}
prob_col = "P(N_{\\text{opp}} \ge 4)"
int_cols = ["N_{\\text{opp}}"]

for col in table_for_paper.columns:
    if col == prob_col:
        # User requested exactly 6 decimals for the probability
        format_dict[col] = "{:.6f}"

    elif col in int_cols:
        format_dict[col] = "{:.0f}"
    
    elif table_for_paper[col].dtype.kind in 'ifc':  # If column is numeric
        # Logic: Calculate decimals needed so the median value has 3 sig figs
        median_val = table_for_paper[col].abs().median()
        
        if pd.isna(median_val) or median_val == 0:
            format_dict[col] = "{:.3f}"
        else:
            # Order of magnitude: floor(log10(val))
            # Decimals = (SigFigs - 1) - magnitude
            magnitude = np.floor(np.log10(median_val))
            decimals = int(max(0, (3 - 1) - magnitude))
            format_dict[col] = "{:." + str(decimals) + "f}"

# 4. Generate LaTeX Outputs via Styler
# We split the dataframe based on T_J, then apply styling
tj_mask = table_for_paper["T_J"].astype(float) < 3

# Helper to stylize and print correctly
def print_styled_latex(df_subset, label):
    # 1. Initialize the styler
    # 2. Apply formatting and replace NaNs
    # 3. Hide the index (this replaces index=False)
    styler = (
        df_subset.style
        .format(format_dict, na_rep="")
        .hide(axis='index')
    )
    
    print(f"% --- Table for {label} ---")
    # In recent Pandas versions, use hrules=True for booktabs (standard for papers)
    print(styler.to_latex(hrules=True))
    print("\n")

# Generate the outputs
tj_mask = table_for_paper["T_J"].astype(float) < 3.02
print_styled_latex(table_for_paper[tj_mask], "T_J < 3.02")
print_styled_latex(table_for_paper[~tj_mask], "T_J >= 3.02")
# print_styled_latex(table_for_paper, "ALL")


/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_45126/3799800816.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  table_for_paper = pd.concat([


% --- Table for T_J < 3.02 ---
\begin{tabular}{lrrrrrrrrl}
\toprule
Object & a & e & i & H_V & T_J & N_{\text{opp}} & P(N_{\text{opp}} \ge 4) & E[N_{\text{opp}}] & Status \\
\midrule
P/2025 UX109 (Ye) & 3.80 & 0.324 & 3.18 & 15.0 & 2.98 &  & 0.999983 & 20.1 & Comet \\
2001 BV70 & 3.59 & 0.426 & 4.33 & 15.1 & 2.95 & 1 & 0.999983 & 19.1 & Suspected Comet, Neg. (Pre)covery \\
P/2022 BV9 (Lemmon) & 4.36 & 0.235 & 11.93 & 13.8 & 2.93 &  & 0.999983 & 37.7 & Comet \\
2017 QN84 & 3.77 & 0.344 & 12.07 & 15.1 & 2.94 & 4 & 0.999982 & 12.7 & Comet \\
(457175) 2008 GO98 & 3.97 & 0.279 & 15.56 & 12.9 & 2.93 & 7 & 0.999981 & 23.8 & Comet \\
2010 RH69 & 4.53 & 0.057 & 11.55 & 13.9 & 2.97 & 3 & 0.999981 & 18.8 & Sam? \\
2019 OE31 & 4.38 & 0.100 & 5.22 & 14.7 & 3.01 & 1 & 0.999981 & 16.5 & "Vacationing Centaur" \\
(323137) 2003 BM80 & 4.23 & 0.188 & 5.81 & 13.6 & 2.99 & 7 & 0.999968 & 21.7 & Comet \\
2018 BJ11 & 4.18 & 0.225 & 3.42 & 15.6 & 2.99 & 1 & 0.999467 & 10.3 & Comet \\
489P/Denning & 4.42 & 0.6

In [ ]:
#
# FALSE POSITIVES TABLE
#

# 1. Prepare the raw data
# Combine main table and comet table
table_for_paper = pd.concat([
    for_paper[for_paper["Principal_desig"].isin(objects_fp)][["Object","a","e","i","H","TJ","Num_opps","prob","exp_Num_opps"]],
    # cometmerge[cometmerge["Designation_and_name"].isin(discoveries_in_comet_file)][["Designation_and_name","a","e","i","H","TJ","Num_opps","prob","exp_Num_opps"]].rename(columns={"Designation_and_name": "Object"})
], ignore_index=True)

# Merge metadata and sort
table_for_paper = table_for_paper.merge(statuses[["Object","Status"]], on="Object", how="left")
table_for_paper.sort_values(by=["prob"], ascending=False, inplace=True)

# 2. Define column renaming
renames = {
    "TJ": "T_J",
    "prob": "P(N_{\\text{opp}} \ge 4)",
    "Num_opps": "N_{\\text{opp}}",
    "exp_Num_opps": "E[N_{\\text{opp}}]",
    "H": "H_V"
}
table_for_paper = table_for_paper.rename(columns=renames)

# 3. Build the Formatter Dictionary
format_dict = {}
prob_col = "P(N_{\\text{opp}} \ge 4)"
int_cols = ["N_{\\text{opp}}"]

for col in table_for_paper.columns:
    if col == prob_col:
        # User requested exactly 6 decimals for the probability
        format_dict[col] = "{:.6f}"

    elif col in int_cols:
        format_dict[col] = "{:.0f}"
    
    elif table_for_paper[col].dtype.kind in 'ifc':  # If column is numeric
        # Logic: Calculate decimals needed so the median value has 3 sig figs
        median_val = table_for_paper[col].abs().median()
        
        if pd.isna(median_val) or median_val == 0:
            format_dict[col] = "{:.3f}"
        else:
            # Order of magnitude: floor(log10(val))
            # Decimals = (SigFigs - 1) - magnitude
            magnitude = np.floor(np.log10(median_val))
            decimals = int(max(0, (3 - 1) - magnitude))
            format_dict[col] = "{:." + str(decimals) + "f}"

# 4. Generate LaTeX Outputs via Styler
# We split the dataframe based on T_J, then apply styling
tj_mask = table_for_paper["T_J"].astype(float) < 3

# Helper to stylize and print correctly
def print_styled_latex(df_subset, label):
    # 1. Initialize the styler
    # 2. Apply formatting and replace NaNs
    # 3. Hide the index (this replaces index=False)
    styler = (
        df_subset.style
        .format(format_dict, na_rep="")
        .hide(axis='index')
    )
    
    print(f"% --- Table for {label} ---")
    # In recent Pandas versions, use hrules=True for booktabs (standard for papers)
    print(styler.to_latex(hrules=True))
    print("\n")

# Generate the outputs
tj_mask = table_for_paper["T_J"].astype(float) < 3
# print_styled_latex(table_for_paper[tj_mask], "T_J < 3")
# print_styled_latex(table_for_paper[~tj_mask], "T_J >= 3")
print_styled_latex(table_for_paper, "ALL")


% --- Table for ALL ---
\begin{tabular}{lrrrrrrrrl}
\toprule
Object & a & e & i & H_V & T_J & N_{\text{opp}} & P(N_{\text{opp}} \ge 4) & E[N_{\text{opp}}] & Status \\
\midrule
2009 US21 & 2.61 & 0.289 & 8.19 & 16.3 & 3.33 & 1 & 0.999983 & 16.9 & Likely Mislinkage \\
2014 DZ209 & 2.44 & 0.162 & 1.27 & 17.2 & 3.48 & 1 & 0.999983 & 16.8 & Likely Mislinkage \\
2009 AN42 & 2.60 & 0.165 & 1.35 & 16.7 & 3.39 & 1 & 0.999983 & 17.7 & Likely Mislinkage \\
2009 VW92 & 2.24 & 0.202 & 8.58 & 16.6 & 3.59 & 1 & 0.999983 & 18.1 & Likely Mislinkage \\
2015 FH479 & 2.88 & 0.066 & 7.84 & 17.2 & 3.28 & 1 & 0.999964 & 10.9 & Likely Mislinkage \\
(826830) 2024 SG49 & 2.54 & 0.197 & 3.96 & 17.4 & 3.41 & 10 & 0.999963 & 13.7 &  \\
2025 WW35 & 2.54 & 0.191 & 4.34 & 17.6 & 3.41 & 1 & 0.999937 & 11.3 & Found 1 night in ITF, need to find 2nd \\
2025 HB52 & 2.80 & 0.072 & 4.90 & 17.6 & 3.31 & 3 & 0.999930 & 10.8 & Linked with two prior opps in ITF \\
2019 GX101 & 2.75 & 0.031 & 3.91 & 18.2 & 3.34 & 1 & 0.998886 & 